# Promote, roll back and reproduce

Linden must separate a model artifact from the version currently used by the next batch job. This worked reference demonstrates aliases, validation tags and explicit application audit events.

Run all cells top to bottom. **Colab:** connect a fresh CPU / Python 3.12 runtime; the first cell clones the pinned GitHub source and installs its locked runtime dependencies. **Local:** run `uv sync --locked` from `module-a` and select that `.venv` interpreter. See [Colab and environment guidance](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/colab.md). Every run creates a new temporary tracking directory and prints its path; download evidence before ending a hosted runtime. No API key is needed. These are complete worked references; independent practices have separate starting states. IDs and timing vary.

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'eec754e0b1379fe8bb01e23543b92e036b4c6b62', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    for name, version in expected.items():
        if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version:
            raise RuntimeError(f'{name} is already loaded at another version. Delete/reconnect the runtime, then run setup first.')
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: eec754e0b1379fe8bb01e23543b92e036b4c6b62 state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-m1aq0azo


## Register and approve a baseline
A new version begins pending. Approval checks guard and the 5-kWh teaching gate. Tags describe evidence; active is a mutable name pointing to a version.

In [2]:
ws=Workspace(state/'tracking')
approved=ws.run(Config(alpha=1,range_guard=True),name='approved')
v1=register(ws,approved['run_id'])
approve(ws,v1,'Guard contracts and baseline validation RMSE checked')
print(activate(ws,v1,'Initial approved version'))
print(ws.client.get_model_version(MODEL_NAME,v1))

2026/10/19 14:42:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:18 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:18 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:18 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:18 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 14:42:18 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:18 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:18 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Successfully registered model 'dev.linden-demand'.
Created version '1' of model 'dev.linden-demand'.


{'timestamp_utc': '2026-10-19T12:42:18.887203+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': None, 'new_version': '1', 'run_id': '196a613ba87b403aa3e5bcf5b1a54f33', 'reason': 'Initial approved version'}
<ModelVersion: aliases=['active'], creation_timestamp=1792413738869, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1792413738869, metrics=None, model_id=None, name='dev.linden-demand', params=None, run_id='196a613ba87b403aa3e5bcf5b1a54f33', run_link=None, source='models:/m-8a615efaffe14cf1b2330653300dd16f', status='READY', status_message=None, tags={'validation_evidence': 'Guard contracts and baseline validation RMSE checked',
 'validation_status': 'approved'}, user_id=None, version=1>


## Exercise a rollback scenario
A trial passes the original 5-kWh gate. A new acceptance brief tightens validation RMSE to 1.6 kWh; the alpha=10 trial fails it while the original alpha=1 passes. Record this reason and roll back the alias without deleting either model.

In [3]:
trial=ws.run(Config(alpha=10,range_guard=True),name='trial')
v2=register(ws,trial['run_id'])
approve(ws,v2,'Passed original 5-kWh teaching gate')
activate(ws,v2,'Trial before stricter acceptance review')
assert trial['metrics']['rmse_kwh']>1.6 and approved['metrics']['rmse_kwh']<1.6
ws.client.set_model_version_tag(MODEL_NAME,v2,'incident','Fails revised 1.6-kWh gate')
event=activate(ws,v1,'Rollback: trial fails revised 1.6-kWh acceptance')
print(event)
assert event['previous_version']==v2 and event['new_version']==v1

/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/19 14:42:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a v

Registered model 'dev.linden-demand' already exists. Creating a new version of this model...


Created version '2' of model 'dev.linden-demand'.


{'timestamp_utc': '2026-10-19T12:42:19.076364+00:00', 'model': 'dev.linden-demand', 'alias': 'active', 'previous_version': '2', 'new_version': '1', 'run_id': '196a613ba87b403aa3e5bcf5b1a54f33', 'reason': 'Rollback: trial fails revised 1.6-kWh acceptance'}


## Verify the next inference job
A rollback is only useful if inference uses it. The helper resolves the alias once and pins its version for this batch.

In [4]:
output,inference=batch_predict(ws)
print(inference)
print((ws.path/'audit.jsonl').read_text())
assert inference['resolved_version']==v1
assert ws.client.get_model_version(MODEL_NAME,v1).run_id==approved['run_id']

{'resolved_version': '1', 'run_id': '196a613ba87b403aa3e5bcf5b1a54f33', 'model': 'dev.linden-demand', 'rows': 24, 'elapsed_seconds': 0.008380333019886166, 'rows_per_second': 2863.8480049717646, 'complete_within_60s': True, 'input_sha256': '5b74deb237ea9a085666ab0b3f865db8c7c996b935b38b24e829567d89142ee6'}
{"timestamp_utc": "2026-10-19T12:42:18.887203+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": null, "new_version": "1", "run_id": "196a613ba87b403aa3e5bcf5b1a54f33", "reason": "Initial approved version"}
{"timestamp_utc": "2026-10-19T12:42:19.070133+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": "1", "new_version": "2", "run_id": "73acbb29faac45beaf4f1515bab262a3", "reason": "Trial before stricter acceptance review"}
{"timestamp_utc": "2026-10-19T12:42:19.076364+00:00", "model": "dev.linden-demand", "alias": "active", "previous_version": "2", "new_version": "1", "run_id": "196a613ba87b403aa3e5bcf5b1a54f33", "reason": "Rollback

## Reproduce the approved run
This checks source, data, config and environment before comparing RMSE. A new store is sufficient for a state replay; the clean-environment procedure additionally restores a source archive and a new venv.

In [5]:
report=reproduce(ws,approved['run_id'],state/'reproduced')
print(report)
assert report['passed'] and report['rmse_delta']<=1e-6
manifest=json.loads((ws.path/'evidence'/approved['run_id']/'manifest.json').read_text())
print({k:manifest[k] for k in ['run_id','code_revision','data_sha256','config','model_uri']})
print('Recorded installed distributions:',len(manifest['environment']['packages']))

2026/10/19 14:42:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 14:42:19 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 14:42:19 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:19 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 14:42:19 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026-10-19 14:42:19 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 14:42:19 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/10/19 14:42:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


{'original_run_id': '196a613ba87b403aa3e5bcf5b1a54f33', 'reproduced_run_id': '940095dae24d45e18fc8ae5cca0527be', 'rmse_delta': 0.0, 'tolerance': 1e-06, 'passed': True, 'lineage_matched': True}
{'run_id': '196a613ba87b403aa3e5bcf5b1a54f33', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'data_sha256': '97fd0648a8277e2ae88f90bb20389745667df3df6b363beef23787ff427e5e54', 'config': {'alpha': 1, 'autolog': False, 'range_guard': True}, 'model_uri': 'models:/m-8a615efaffe14cf1b2330653300dd16f'}
Recorded installed distributions: 119
